# 🤗 Hugging Face Transformers - Sentiment Analysis Hands-on

Welcome to this hands-on notebook on **Hugging Face Transformers**, where you'll learn how to perform **Sentiment Analysis** on text using pre-trained Transformer models.

---

## What You’ll Learn

By the end of this notebook, you’ll be able to:

* Understand how to use Hugging Face's `pipeline` for quick NLP tasks  
* Load and use a specific model (`distilbert-base-uncased-finetuned-sst-2-english`)  
* Tokenize input manually and pass it through the model  
* Understand what happens under the hood in the inference pipeline  
* Convert raw model output (logits) to probabilities and class labels

---

## Hands-on Task: Sentiment Analysis

We will use a **Transformer model** fine-tuned for sentiment classification to analyze whether a given sentence expresses a **positive** or **negative** sentiment.

---

## Structure of this Notebook

### Part 1: Simple Version
- Using `pipeline()` for quick sentiment prediction
- Abstracts all steps internally

### Part 2: Under the Hood
- Manual tokenization
- Direct model interaction
- Step-by-step breakdown of each component

---

Let’s get started!

# Part 1: Sentiment Analysis with Hugging Face Transformers (Simple Version)

In this section, we'll use the `pipeline` utility from Hugging Face Transformers, which abstracts away the complexity and lets you do sentiment analysis in just a few lines of code.


In [ ]:
# Install transformers if you haven't
!pip install -q transformers

In [2]:
from transformers import pipeline

# Create a text classification pipeline (sentiment analysis)
classifier = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")

# Test it on a sentence
text = "I absolutely love the Hugging Face library!"
result = classifier(text)

print(result)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998668432235718}]


In [4]:
text = "I absolutely hate the Hugging Face library!"
result = classifier(text)
print(result)

[{'label': 'NEGATIVE', 'score': 0.9933987855911255}]


## Explanation:
- `pipeline("text-classification")` loads a default sentiment analysis model.
- We specified `"distilbert-base-uncased-finetuned-sst-2-english"` — a lightweight version of BERT trained on SST-2.
- It returns a label (`POSITIVE` or `NEGATIVE`) and a confidence score.

This is the easiest way to get started!

# Part 2: Sentiment Analysis with Hugging Face Transformers (Under the Hood)

In this section, we'll go deeper into how the pipeline works under the hood. We'll manually tokenize the text, pass it through the model, and decode the result step-by-step.

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import numpy as np

In [6]:
# Load model and tokenizer manually
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

## Step 1: Tokenize the text
In this step, we use the **Hugging Face AutoTokenizer** to convert raw text into numerical representations that the model can understand.

A **tokenizer** is responsible for:
1. Splitting the input text into smaller units called tokens (often words, subwords, or even characters).
2. Mapping tokens to numerical IDs using a predefined vocabulary associated with the pretrained model.
3. Creating attention masks to inform the model which tokens are actual input and which are just padding (if any).

In [8]:
text = "I absolutely love the Hugging Face library!"

# Tokenize and return PyTorch tensors
inputs = tokenizer(text, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101,  1045,  7078,  2293,  1996, 17662,  2227,  3075,   999,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


the **tokenizer** performs several operations:

1. Tokenization:
> Converts "I absolutely love the Hugging Face library!" into token pieces, like ["i", "absolutely", "love", "the", "hugging", "face", "library", "!"].

2. Subword Splitting:
> For rare or compound words (like "HuggingFace"), it may split them further, e.g., ["hugging", "face"] or ["hugging", "##face"] depending on the model.

3. Conversion to IDs:
> Each token is converted into an integer index using the model's vocabulary. Example: "love" → 2293.

4. Attention Mask Creation:
> Indicates which tokens should be attended to (1 for real tokens, 0 for padding).

## Step 2: Pass inputs to the model

In this step, we take the tokenized input and feed it into the pretrained **DistilBERT** model to compute raw prediction values known as logits.

`model(**inputs)`
- This is a forward pass through the Transformer model.
We pass the input_ids and attention_mask from the tokenizer as keyword arguments.
- Internally, the model processes the tokens, attends to the most relevant parts of the sentence using self-attention, and generates contextual embeddings.

`outputs.logits`
- The model returns a dictionary-like object with several outputs (depending on the model).
- For classification tasks like sentiment analysis, we’re interested in the logits — a raw, unnormalized score for each class.

In [9]:
# Get model output (logits)
outputs = model(**inputs)
logits = outputs.logits

print("Logits:", logits)

Logits: tensor([[-4.2856,  4.6383]], grad_fn=<AddmmBackward0>)


Logits are the output of the model before applying any activation function like softmax. They represent the model’s confidence in each class: `tensor([[-4.2856, 4.6383]])`

The shape of the logits is [batch_size, num_labels]. In this case, it's [1, 2] because:
- Batch size = 1 (we only passed one sentence)
- Number of classes = 2 (positive or negative sentiment)

In this example:
- `-4.2856` is the score for class 0 (likely "negative")
- `4.6383` is the score for class 1 (likely "positive")

These scores are not probabilities yet — they will be passed through a softmax function in the next step to get normalized probabilities.

## Step 3: Convert logits to probabilities

After obtaining raw logits from the model, we apply the **softmax function** to transform them into interpretable probabilities.

Softmax is a mathematical function that converts raw scores (logits) into probabilities that:

- Are **bounded between 0 and 1**
- **Sum to 1** across all classes

In [10]:
probs = F.softmax(logits, dim=-1)
print("Probabilities:", probs)

Probabilities: tensor([[1.3315e-04, 9.9987e-01]], grad_fn=<SoftmaxBackward0>)


## Step 4: Decode the result

We find the class with the highest probability and map it back to its human-readable label.


In [11]:
# Get predicted class
predicted_class_id = torch.argmax(probs).item()
label = model.config.id2label[predicted_class_id]

print(f"Predicted label: {label}, Score: {probs[0][predicted_class_id]:.4f}")

Predicted label: POSITIVE, Score: 0.9999


## Summary

- We manually tokenized the input text.
- Passed it through the pre-trained Transformer model.
- Extracted raw logits → probabilities → human-readable label.

This approach gives you more control and understanding of each stage in the Transformer pipeline.